# Elevator fleet — monitoring at fleet scale

The cloud half of lecture 10. Local EPL controls **one building** in milliseconds;
this reads **the whole fleet** in minutes, and answers questions the controller
cannot even ask.

**NOTE**: run the early part of `1_fleet_simulator.ipynb` first.

Four queries, `Q.10.1` to `Q.10.4`. The last two are the ones worth the hour.

## The session, and the Kafka connector

In [1]:
import os, json
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, from_json, window
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, TimestampType)

import glob, pyspark

# the Kafka connector is not part of Spark: it is fetched from Maven, and its
# coordinate has to match *this* Spark and the Scala it was built against.
# Derive both instead of hard-coding them, so the notebook cannot silently
# disagree with the image it is running in.
jars  = glob.glob(pyspark.__path__[0] + '/jars/scala-library-2.13*')
scala = '2.13' if jars else '2.12'
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages org.apache.spark:spark-sql-kafka-0-10_%s:%s pyspark-shell'
    % (scala, pyspark.__version__))
print('spark', pyspark.__version__, '/ scala', scala)

spark = (SparkSession.builder
         .master("local[*]")
         .appName("elevator-fleet")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark

spark 3.5.0 / scala 2.12


The timezone is pinned to UTC on purpose. Every timestamp in this module is UTC, and
a fleet spanning three continents is the last place you want the session timezone to
decide what a window boundary means.

In [2]:
servers = "kafka:9092"
topic_events = "elevator-events"
topic_faults = "elevator-faults"

schema = StructType([
    StructField("event",     StringType()),
    StructField("unitId",    StringType()),
    StructField("ts",        TimestampType()),
    StructField("car",       StringType()),
    StructField("floor",     IntegerType()),
    StructField("servedDir", StringType()),
])

In [3]:
raw = (spark.readStream
       .format("kafka")
       .option("kafka.bootstrap.servers", servers)
       .option("subscribe", topic_events)
       .option("startingOffsets", "earliest")
       .load())

events = (raw
          .select(from_json(col("value").cast("string"), schema).alias("v"))
          .select("v.*"))
doors = events.where(col("event") == "DoorOpened")
doors.printSchema()

root
 |-- event: string (nullable = true)
 |-- unitId: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- car: string (nullable = true)
 |-- floor: integer (nullable = true)
 |-- servedDir: string (nullable = true)



## The static side

A unit belongs to a building, and a building sits in a region. That mapping does not
stream — it is reference data, and joining a stream against it is a **stream-static
join**: no watermark, no state, every micro-batch re-reads the small side.

In [4]:
# the same three lines as the simulator: one source for the fleet
REGIONS = {"EU": "Europe/Milan", "US": "America/New_York", "JP": "Asia/Tokyo"}
UNITS   = [f"{r}-{i:03d}" for r in REGIONS for i in range(1, 9)]

units = spark.createDataFrame([
    {"unitId": u,
     "buildingId": f"B{u.split(chr(45))[1]}",
     "region": REGIONS[u.split(chr(45))[0]]}
    for u in UNITS])
units.show(5, False)

+----------+------------+------+
|buildingId|region      |unitId|
+----------+------------+------+
|B001      |Europe/Milan|EU-001|
|B002      |Europe/Milan|EU-002|
|B003      |Europe/Milan|EU-003|
|B004      |Europe/Milan|EU-004|
|B005      |Europe/Milan|EU-005|
+----------+------------+------+
only showing top 5 rows



---
## Q.10.1 — how many doors, per unit, per minute

In [5]:
q101 = (doors
        .withWatermark("ts", "2 minutes")
        .groupBy(window(col("ts"), "1 minute").alias("w"), col("unitId"))
        .count()
        .writeStream.format("memory").outputMode("update")
        .queryName("q101").start())

### Go to the simulator: section 1) Background traffic

Run it there, then come back and run the cell below — **only that one**. The cell
above has started a query that is still running; you never re-run it.

In [6]:
spark.sql("""SELECT w.start AS minute, unitId, count
             FROM q101 ORDER BY minute, unitId""").show(12, False)

if spark.table("q101").count() == 0:
    print("empty — either section 1) has not been produced yet, or the batch has")
    print("not landed: wait a second and re-run this cell. An empty table never")
    print("means the query failed; a failed query raises, and q101.exception() says so.")

+-------------------+------+-----+
|minute             |unitId|count|
+-------------------+------+-----+
|2026-10-15 08:00:00|EU-001|1    |
|2026-10-15 08:00:00|EU-002|1    |
|2026-10-15 08:00:00|EU-003|1    |
|2026-10-15 08:00:00|EU-004|1    |
|2026-10-15 08:00:00|EU-005|1    |
|2026-10-15 08:00:00|EU-006|1    |
|2026-10-15 08:00:00|EU-007|1    |
|2026-10-15 08:00:00|EU-008|1    |
|2026-10-15 08:00:00|JP-001|6    |
|2026-10-15 08:00:00|JP-002|6    |
|2026-10-15 08:00:00|JP-003|6    |
|2026-10-15 08:00:00|JP-004|6    |
+-------------------+------+-----+
only showing top 12 rows



### The same minute can appear twice, and it is not a bug

The sink is `memory` and the output mode is `update`, and **a memory sink in update
mode is a log of updates, not a table**. Every time a `(window, unitId)` group is
updated in a new micro-batch, another row for that same key is appended, carrying the
new running count. Ask it:

In [7]:
spark.sql("""SELECT w.start AS minute, unitId,
                    COUNT(*) AS rows_in_sink, MAX(count) AS final
             FROM q101 GROUP BY 1, 2 HAVING COUNT(*) > 1
             ORDER BY 1, 2""").show(10, False)

+------+------+------------+-----+
|minute|unitId|rows_in_sink|final|
+------+------+------------+-----+
+------+------+------------+-----+



**On your run this may well be empty** — whether a minute of one unit arrives in one
micro-batch or two is decided by the consumer, not by the data and not by you. Which
is exactly why the next cell is not optional: a query whose correctness depends on
batch boundaries is a query that will be wrong on someone else's laptop.

Counts inside a group only grow, so the last value is the largest. One row per key:

In [8]:
spark.sql("""CREATE OR REPLACE TEMP VIEW q101_final AS
             SELECT w, unitId, MAX(count) AS count
             FROM q101 GROUP BY w, unitId""")

# everything below reads q101_final. Reading q101 directly and summing would
# count the intermediate updates as if they were events.
spark.sql("SELECT SUM(count) AS door_events FROM q101_final").show()

+-----------+
|door_events|
+-----------+
|        384|
+-----------+



---
## Q.10.2 — the same doors, per region

One line different from `Q.10.1`: the join against the reference data, and a group by
`region` instead of `unitId`.

In [9]:
q102 = (doors.join(units, on="unitId")
        .withWatermark("ts", "2 minutes")
        .groupBy(window(col("ts"), "1 minute").alias("w"), col("region"))
        .count()
        .writeStream.format("memory").outputMode("update")
        .queryName("q102").start())

In [12]:
# same sink, same output mode, same precaution as for q101
spark.sql("""CREATE OR REPLACE TEMP VIEW q102_final AS
             SELECT w, region, MAX(count) AS count
             FROM q102 GROUP BY w, region""")

spark.sql("""SELECT w.start AS minute, region, count
             FROM q102_final ORDER BY minute, region""").show(20, False)

+-------------------+----------------+-----+
|minute             |region          |count|
+-------------------+----------------+-----+
|2026-10-15 08:00:00|America/New_York|8    |
|2026-10-15 08:00:00|Asia/Tokyo      |48   |
|2026-10-15 08:00:00|Europe/Milan    |8    |
|2026-10-15 08:01:00|America/New_York|8    |
|2026-10-15 08:01:00|Asia/Tokyo      |32   |
|2026-10-15 08:01:00|Europe/Milan    |16   |
|2026-10-15 08:02:00|America/New_York|8    |
|2026-10-15 08:02:00|Asia/Tokyo      |16   |
|2026-10-15 08:02:00|Europe/Milan    |48   |
|2026-10-15 08:03:00|America/New_York|16   |
|2026-10-15 08:03:00|Asia/Tokyo      |8    |
|2026-10-15 08:03:00|Europe/Milan    |48   |
|2026-10-15 08:04:00|America/New_York|32   |
|2026-10-15 08:04:00|Asia/Tokyo      |8    |
|2026-10-15 08:04:00|Europe/Milan    |16   |
|2026-10-15 08:05:00|America/New_York|48   |
|2026-10-15 08:05:00|Asia/Tokyo      |8    |
|2026-10-15 08:05:00|Europe/Milan    |8    |
+-------------------+----------------+-----+



### Now add the totals up

Per region the load swings by a factor of six across six minutes. Add the three
regions together for each minute and look at the range of **that**.

In [13]:
spark.sql("""SELECT w.start AS minute, SUM(count) AS fleet_total
             FROM q102_final GROUP BY w.start ORDER BY minute""").show(20, False)

+-------------------+-----------+
|minute             |fleet_total|
+-------------------+-----------+
|2026-10-15 08:00:00|64         |
|2026-10-15 08:01:00|56         |
|2026-10-15 08:02:00|72         |
|2026-10-15 08:03:00|72         |
|2026-10-15 08:04:00|56         |
|2026-10-15 08:05:00|64         |
+-------------------+-----------+



**Rush hour is local.** Each region has a peak and the fleet as a whole barely
notices — the aggregate curve is nearly flat while every region on it is not.

That is the most concrete argument available for partitioning by **region** rather
than by unit: `unitId` gives you an almost perfectly uniform key and a flat
consumer load, which is what you want for throughput — and it also guarantees that
no consumer ever sees a region whole, which is what you want for a rush-hour
dashboard. The two goals disagree, and the data is what makes the disagreement
visible.

---
## Q.10.3 — bunching, and what it costs to lose `->`

In EPL, lecture 7 wrote it in three lines:

```
@name('bunching')
select a.car as firstCar, b.car as secondCar, a.floor as floor
from pattern [
  every a=DoorOpened
    -> b=DoorOpened(floor=a.floor, servedDir=a.servedDir, car != a.car)
       where timer:within(20 sec)
];
```

**Structured Streaming has no sequence operator.** There is no `->`. So the same
question becomes a **stream-stream self-join**, with a watermark on each side and the
ordering written by hand as a temporal predicate.

In [14]:
a = doors.withWatermark("ts", "30 seconds").alias("a")
b = doors.withWatermark("ts", "30 seconds").alias("b")

q103 = (a.join(b, expr("""
            a.unitId = b.unitId
        AND a.floor  = b.floor
        AND a.servedDir = b.servedDir
        AND a.car   <> b.car
        AND b.ts >  a.ts
        AND b.ts <= a.ts + interval 20 seconds"""))
        .select(col("a.unitId").alias("unitId"),
                col("a.floor").alias("floor"),
                col("a.servedDir").alias("dir"),
                col("a.car").alias("firstCar"),
                col("b.car").alias("secondCar"),
                col("a.ts").alias("firstTs"),
                col("b.ts").alias("secondTs"))
        .writeStream.format("memory").outputMode("append")
        .queryName("q103").start())

### Go to the simulator: section 2) One bunching

Three configurations go in. **Predict how many rows come out, and which clause cuts
each of the others**, before you look.

In [15]:
spark.sql("SELECT * FROM q103 ORDER BY firstTs").show(10, False)

if spark.table("q103").count() == 0:
    print("empty — and here that can also be legitimate: the join writes in append")
    print("mode, so a row is emitted only once the watermark guarantees no partner")
    print("can still arrive. Producing the next section pushes it forward.")

+------+-----+---+--------+---------+-------------------+-------------------+
|unitId|floor|dir|firstCar|secondCar|firstTs            |secondTs           |
+------+-----+---+--------+---------+-------------------+-------------------+
|EU-003|3    |UP |A       |B        |2026-10-15 08:07:00|2026-10-15 08:07:08|
+------+-----+---+--------+---------+-------------------+-------------------+



### Read the differences, not the result

Count the things the EPL version did not have to say:

* **`b.ts > a.ts`** — `->` means *followed by*. A join is symmetric, so without this
  line every bunching comes out twice, once in each order.
* **two watermarks** — a join between two streams needs a bound on how long each side
  waits, or the state grows forever. In EPL `timer:within(20 sec)` was that bound
  *and* the business rule at once; here they are two separate decisions.
* **`append` output mode** — a stream-stream join cannot emit under `update`, because
  a row is only final once the watermark says no partner can still arrive.

It works, it is more verbose, and it is less expressive. That is the operational
difference between a CEP engine and a distributed stream processor, and it needs no
theory: you have just written both.

---
## Q.10.4 — the unit that reconnects, and the events nobody misses

Before running the next section, read the watermark — it is what decides the answer.

In [16]:
print("watermark now:", q101.lastProgress["eventTime"].get("watermark"))

watermark now: 2026-10-15T08:07:06.000Z


### Go to the simulator: section 3) A unit that reconnects

Six door events come back at once, with event times in minutes 6 and 7 — already in
the past.

**Predict how many of the six are counted.** Then look.

In [17]:
spark.sql("""SELECT w.start AS minute, count FROM q101_final
             WHERE unitId = 'JP-005' ORDER BY minute""").show(20, False)

+-------------------+-----+
|minute             |count|
+-------------------+-----+
|2026-10-15 08:00:00|6    |
|2026-10-15 08:01:00|4    |
|2026-10-15 08:02:00|2    |
|2026-10-15 08:03:00|1    |
|2026-10-15 08:04:00|1    |
|2026-10-15 08:05:00|1    |
|2026-10-15 08:07:00|3    |
+-------------------+-----+



Now the instrumentation — and reading it takes one precaution. `q.lastProgress` is
the *last* micro-batch, and with a live source Spark keeps running batches when there
is nothing to read, so by the time you look it is almost always an empty one: zero
rows, zero dropped. What you want is the last batch that actually ingested something.

In [18]:
def last_fed_batch(q):
    """The most recent micro-batch that ingested rows, not merely the last one."""
    fed = [p for p in q.recentProgress if p["numInputRows"] > 0]
    return fed[-1] if fed else None

p = last_fed_batch(q101)
print("batch id                     :", p["batchId"])
print("input rows in that batch     :", p["numInputRows"])
print("watermark in force for it    :", p["eventTime"].get("watermark"))
for so in p["stateOperators"]:
    print("numRowsDroppedByWatermark    :", so["numRowsDroppedByWatermark"])

batch id                     : 10
input rows in that batch     : 6
watermark in force for it    : 2026-10-15T08:07:06.000Z
numRowsDroppedByWatermark    : 1


Note *"in force for it"*: a batch reports the watermark it **used**, computed from the
data seen before it — not the one its own rows produce. That is why this batch and the
cell above can disagree, and the one that decided the fate of the six events is this
one.

### Two things to take away, and the second is a trap

**The backlog is split by where the watermark happened to be.** The events whose
window had already closed are gone — silently, with no error and no dead-letter. The
events whose window was still open are counted as if nothing had happened. Nothing
about the unit changed; only the arrival time did.

**`numRowsDroppedByWatermark` does not count events.** It counts **state rows** —
one per `(window, key)` group. Several dropped events belonging to the same unit and
the same minute are pre-aggregated inside the batch and reported as **one**. If you
use that metric as "events lost", you will under-count, and by a factor that depends
on how the backlog happens to be distributed.

Ask the real question: for a fleet with a long tail of reconnecting units, **what is
the watermark actually buying?** Widen it and state grows and results arrive later;
narrow it and you quietly lose the units that most need watching, because a lift that
loses its network is not a lift that is doing well.

---
## The sizing arithmetic

Now make the numbers yourself, from what this run actually measured rather than from
a slide.

In [19]:
# q101_final, never q101: one row per key, or the intermediate updates get counted
rows = spark.sql("SELECT SUM(count) AS n FROM q101_final").first()["n"]
span = spark.sql("""SELECT (MAX(w.start) - MIN(w.start)) AS s FROM q101_final""").first()["s"]
n_units = spark.sql("SELECT COUNT(DISTINCT unitId) AS u FROM q101_final").first()["u"]
minutes = span.total_seconds() / 60 + 1

per_unit_per_min = rows / n_units / minutes
print(f"{rows} door events, {n_units} units, {minutes:.0f} minutes of fleet time")
print(f"-> {per_unit_per_min:.2f} door events per unit per minute")

393 door events, 24 units, 10 minutes of fleet time
-> 1.64 door events per unit per minute


In [20]:
FLEET = 150_000        # connected units, the order of magnitude of a real operator
BYTES = 250            # a JSON door event, on the wire

per_sec = FLEET * per_unit_per_min / 60
mb_sec  = per_sec * BYTES / 1e6
print(f"{FLEET:,} units  ->  {per_sec:,.0f} events/s  ->  {mb_sec:.2f} MB/s")
print(f"per day        ->  {per_sec*86400/1e6:,.0f} M events, {mb_sec*86400/1000:,.1f} GB")

150,000 units  ->  4,094 events/s  ->  1.02 MB/s
per day        ->  354 M events, 88.4 GB


### Check your denominator before you believe the number

That rate was divided by the **whole** span of fleet time, and the last minutes of
this run contain only the designed sections — one bunching and one reconnecting
unit. Almost nothing happened in them, and they dragged the average down.

Measure the same fleet over the six minutes of background traffic instead.

In [21]:
busy = spark.sql(
    "SELECT SUM(count) AS n, COUNT(DISTINCT unitId) AS u "
    "FROM q101_final WHERE w.start < '2026-10-15 08:06:00'").first()

busy_rate = busy['n'] / busy['u'] / 6
print(f"background only: {busy['n']} events / {busy['u']} units / 6 min")
print(f"-> {busy_rate:.2f} per unit per minute, "
      f"{busy_rate/per_unit_per_min:.1f}x the figure above")
print(f"{FLEET:,} units -> {FLEET*busy_rate/60:,.0f} events/s"
      f" -> {FLEET*busy_rate/60*BYTES/1e6:.2f} MB/s")

background only: 384 events / 24 units / 6 min
-> 2.67 per unit per minute, 1.6x the figure above
150,000 units -> 6,667 events/s -> 1.67 MB/s


**The answer moved by more than half, and nothing about the fleet changed** — only
the window you averaged over. A capacity plan built on the first number and one built
on the second are different clusters.

This is the whole reason to compute a rate rather than quote one. Neither number is
wrong; they answer different questions — *what does this fleet cost on average* and
*what does it cost when it is busy* — and only one of them sizes a cluster.

### The counterintuitive part

Compare that MB/s against what a single broker will do. **The worldwide fleet of a
major operator is a small Kafka problem in bytes.** Which forces the question the
lecture is actually about: if throughput is not what sizes this cluster, what does?

* **retention** — predictive maintenance wants months of door-cycle history to see
  drift, and months of history is a storage decision, not a bandwidth one;
* **partition count** — driven by consumer parallelism, not by MB/s;
* **the key** — `unitId` is almost perfectly uniform, which is the *easy* case;
  the retail lecture is where the data fights back;
* **the long tail** — the reconnecting units of `Q.10.4`, which cost state and
  correctness rather than bandwidth.

Note what the arithmetic above depends on: `per_unit_per_min`, measured from a
24-unit simulation over six minutes. Change the simulator and every number below it
changes. That is the point — the estimate is a **calculation you can audit**, not a
figure on a slide.

---
## Clean up

In [22]:
for q in (q101, q102, q103):
    q.stop()